In [1]:
import re
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()


In [2]:
clean_articles_df = pd.read_parquet("clean_articles_step1.parquet")
clean_articles_df.shape

(153512, 4)

In [3]:
clean_articles_df.head()

,article_id,date,title,content
0,1,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...
1,2,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...
2,3,2023-12-12,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...
3,4,2023-09-07,Zoom Expands AI Offering with AI Companion and...,Zoom Expands AI Offering with AI Companion and...
4,5,2023-08-04,Pro-AI Thinking: Enhancing Industrial Environm...,Pro-AI Thinking: Enhancing Industrial Environm...


In [4]:
boilerplate_patterns = [
    r"skip\s*to\s*content\w*",
    r"watch\s*live\w*",
    r"community\s*calendar\w*",
    r"closings?(?:\s*&\s*|\s+and\s+)?delays?\b",
    r"submit\s*photos?(?:\s*&\s*|\s+and\s+)?videos?\b",
    r"download\s*(?:our\s*)?apps?\b",
    r"sign\s*up\s*for\s*(?:our\s*)?newsletters?\b",
    r"newsletters?\b",
    r"advertis\w*(?:\s+with\s+us)?\b",
    r"send\s*us\s*a\s*news\s*tip\b",
    r"weather\s*radar\b",
    r"election\s*results\b",
    r"home\s*news\s*weather\s*sports(?:\s*\w+){0,20}",
]
bp_regex = re.compile("|".join(boilerplate_patterns), flags=re.IGNORECASE)

def normalize_space(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def topic_clean(title: str, content: str) -> str:
    t = normalize_space(title)
    c = normalize_space(content)

    # 去正文开头重复标题（最多两次）
    for _ in range(2):
        if t and c.lower().startswith(t.lower()):
            c = c[len(t):].strip(" -:|")

    # 删固定模板短语
    c = bp_regex.sub(" ", c)

    # 删导航词连续堆叠片段
    c = re.sub(
        r"(?:\b(home|news|weather|sports|live|video|search|menu|about|contact)\b[\s|/:,-]*){5,}",
        " ",
        c,
        flags=re.IGNORECASE,
    )

    c = re.sub(r"[\r\n\t]+", " ", c)
    c = re.sub(r"\s{2,}", " ", c).strip()
    return c

work_df = clean_articles_df.copy()
work_df["title"] = work_df["title"].fillna("").astype(str).str.strip()
work_df["content"] = work_df["content"].fillna("").astype(str).str.strip()

work_df["content_topic"] = [
    topic_clean(t, c) for t, c in zip(work_df["title"], work_df["content"])
]
work_df["text_raw"] = (work_df["title"] + " " + work_df["content_topic"]).str.strip()
work_df = work_df[work_df["text_raw"].str.len() > 0].reset_index(drop=True)

print("usable docs:", len(work_df))


usable docs: 153512


## Entity extraction model options (organizations + technologies)

推荐优先级：
1. **GLiNER (`urchade/gliner_large-v2.1`)**：支持自定义 label，能直接抽 `organization` 和 `technology`。
2. **spaCy `en_core_web_trf` + EntityRuler**：`ORG` 稳定，`technology` 用规则补充。
3. **`dslim/bert-base-NER`**：`ORG` baseline 快速，`technology` 需要二次规则/词典。

下面代码默认用 GLiNER，并按你的想法先跑 sample，再选择是否跑全量。

In [5]:
from gliner import GLiNER

TEXT_COL = "text_raw"  # 也可以改成 content_topic
TARGET_LABELS = ["organization", "technology"]
SAMPLE_SIZE = 120
THRESHOLD = 0.5
MAX_CHARS = 4000

model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")
print("model loaded")

/opt/anaconda3/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

model loaded


In [6]:
def extract_entities_gliner(text, threshold=THRESHOLD, max_chars=MAX_CHARS):
    if not isinstance(text, str) or not text.strip():
        return []

    text = text[:max_chars]
    entities = model.predict_entities(text, TARGET_LABELS, threshold=threshold)

    cleaned = []
    seen = set()
    for ent in entities:
        label = str(ent.get("label", "")).lower().strip()
        mention = str(ent.get("text", "")).strip()
        score = float(ent.get("score", 0))

        if label not in {"organization", "technology"}:
            continue
        if len(mention) < 2:
            continue

        key = (mention.lower(), label)
        if key in seen:
            continue
        seen.add(key)

        cleaned.append({
            "entity": mention,
            "label": label,
            "score": round(score, 4),
        })

    return cleaned

In [7]:
sample_n = min(SAMPLE_SIZE, len(work_df))
sample_df = work_df.sample(n=sample_n, random_state=42).copy()
sample_df["entities"] = sample_df[TEXT_COL].progress_apply(extract_entities_gliner)

sample_entities = (
    sample_df[["title", "entities"]]
    .explode("entities", ignore_index=True)
    .dropna(subset=["entities"])
)

if len(sample_entities):
    sample_entities = pd.concat(
        [sample_entities.drop(columns=["entities"]), sample_entities["entities"].apply(pd.Series)],
        axis=1,
    )

print(f"sample docs: {len(sample_df)}")
print(f"sample extracted entities: {len(sample_entities) if 'sample_entities' in locals() else 0}")
sample_entities.head(15)

  0%|          | 0/120 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 643 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 629 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 764 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/opt/anaconda3/lib/python3.13/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 801 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/opt/anaconda3/l

sample docs: 120
sample extracted entities: 611


,title,entity,label,score
0,Warren Buffett Has Bet Over $156 Billion on Th...,Nasdaq,organization,0.9885
1,Warren Buffett Has Bet Over $156 Billion on Th...,Technology,technology,0.7799
2,Warren Buffett Has Bet Over $156 Billion on Th...,Nasdaq-100,organization,0.7385
3,Hoot Launches AI Integration on its Myopia Man...,Hoot,organization,0.5780
4,Hoot Launches AI Integration on its Myopia Man...,AI,technology,0.9765
5,Hoot Launches AI Integration on its Myopia Man...,"Hoot Health, Inc.",organization,0.8332
6,Hoot Launches AI Integration on its Myopia Man...,Artificial Intelligence,technology,0.9398
7,Twitch resurrects AI Jesus - Israel365 News,Twitch,organization,0.8062
8,Twitch resurrects AI Jesus - Israel365 News,AI Jesus,technology,0.6197
9,Twitch resurrects AI Jesus - Israel365 News,Singularity Group,organization,0.9819


In [8]:
if len(sample_entities):
    display(sample_entities["label"].value_counts())
    display(
        sample_entities.groupby(["label", "entity"]).size().reset_index(name="count")
        .sort_values(["label", "count"], ascending=[True, False])
        .groupby("label")
        .head(20)
    )

label
organization    386
technology      225
Name: count, dtype: int64

,label,entity,count
182,organization,OpenAI,14
102,organization,Google,11
154,organization,Microsoft,10
177,organization,Nvidia,7
16,organization,Amazon,6
15,organization,Alphabet,4
122,organization,Intel,4
5,organization,AP,3
8,organization,AWS,3
23,organization,Apple,3


In [ ]:
# 样本结果满意后，把 RUN_FULL 改成 True
RUN_FULL = False

if RUN_FULL:
    full_df = work_df.copy()
    full_df["entities"] = full_df[TEXT_COL].progress_apply(extract_entities_gliner)

    full_entities = (
        full_df[["title", "entities"]]
        .explode("entities", ignore_index=True)
        .dropna(subset=["entities"])
    )
    full_entities = pd.concat(
        [full_entities.drop(columns=["entities"]), full_entities["entities"].apply(pd.Series)],
        axis=1,
    )

    full_df.to_parquet("entity_extraction_docs.parquet", index=False)
    full_entities.to_parquet("entity_extraction_mentions.parquet", index=False)

    print("saved: entity_extraction_docs.parquet")
    print("saved: entity_extraction_mentions.parquet")
    display(full_entities.head())